# Lab 17 — Cortex AI Cost Monitoring

Track and optimize Cortex AI spending using Snowflake's built-in usage views.

| Item | Detail |
|---|---|
| **Views** | `METERING_DAILY_HISTORY`, `CORTEX_FUNCTIONS_USAGE_HISTORY`, `CORTEX_SEARCH_DAILY_USAGE_HISTORY` |
| **Functions** | `AI_COUNT_TOKENS` for cost estimation |
| **Exam Domain** | 3.0 Gen AI Governance |
| **You'll learn** | Credit tracking, per-function costs, token estimation, cost optimization |


## Step 1 — Environment Setup

> **AI SQL Functions** — This lab uses the preferred `AI_` prefixed functions. 
Equivalent legacy functions: `AI_COUNT_TOKENS` (replaces `SNOWFLAKE.CORTEX.COUNT_TOKENS`).

In [ ]:
USE ROLE DS_ROLE;
USE WAREHOUSE DS_WH;
USE DATABASE GENAI_LEARNING;
USE SCHEMA PUBLIC;

## Step 2 — Cortex Usage Views

| View | What It Tracks |
|---|---|
| `SNOWFLAKE.ACCOUNT_USAGE.METERING_DAILY_HISTORY` | Overall daily credit consumption |
| `SNOWFLAKE.ACCOUNT_USAGE.CORTEX_FUNCTIONS_USAGE_HISTORY` | Per-function Cortex credit usage |
| `SNOWFLAKE.ACCOUNT_USAGE.CORTEX_FUNCTIONS_QUERY_USAGE_HISTORY` | Per-query Cortex costs |
| `SNOWFLAKE.ACCOUNT_USAGE.CORTEX_SEARCH_DAILY_USAGE_HISTORY` | Search service daily costs |
| `SNOWFLAKE.ACCOUNT_USAGE.CORTEX_DOCUMENT_PROCESSING_USAGE_HISTORY` | Document processing costs |


In [ ]:
SELECT
    USAGE_DATE,
    SERVICE_TYPE,
    ROUND(CREDITS_USED, 4) AS CREDITS
FROM SNOWFLAKE.ACCOUNT_USAGE.METERING_DAILY_HISTORY
WHERE SERVICE_TYPE ILIKE '%CORTEX%'
    AND USAGE_DATE >= DATEADD('day', -30, CURRENT_DATE())
ORDER BY USAGE_DATE DESC
LIMIT 20;

In [ ]:
SELECT
    FUNCTION_NAME,
    FUNCTION_TYPE,
    ROUND(SUM(TOKENS), 0) AS TOTAL_TOKENS,
    ROUND(SUM(CREDITS_USED), 6) AS TOTAL_CREDITS,
    COUNT(*) AS CALL_COUNT
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_FUNCTIONS_USAGE_HISTORY
WHERE START_TIME >= DATEADD('day', -30, CURRENT_DATE())
GROUP BY FUNCTION_NAME, FUNCTION_TYPE
ORDER BY TOTAL_CREDITS DESC;

## Step 3 — Search Service Costs

In [ ]:
SELECT
    SERVICE_NAME,
    USAGE_DATE,
    ROUND(CREDITS_USED, 6) AS CREDITS
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_SEARCH_DAILY_USAGE_HISTORY
WHERE USAGE_DATE >= DATEADD('day', -30, CURRENT_DATE())
ORDER BY USAGE_DATE DESC
LIMIT 20;

## Step 4 — Pre-Estimate Costs with COUNT_TOKENS

In [ ]:
SELECT
    'claude-haiku-4-5' AS model,
    COUNT(*) AS row_count,
    SUM(AI_COUNT_TOKENS('ai_complete', 'claude-haiku-4-5', body)) AS total_input_tokens
FROM wiki_articles;

## Step 5 — Cost Optimization Strategies

| Strategy | How |
|---|---|
| Use smaller models | `claude-haiku-4-5` vs `claude-sonnet-4-5` when quality permits |
| Use specialized functions | `AI_SENTIMENT` is cheaper than `AI_COMPLETE` for sentiment |
| Pre-filter with SQL | Only send relevant rows to AI functions |
| Cache results | Store AI outputs in tables instead of recomputing |
| Batch with TRY_COMPLETE | Avoid failed-row overhead in large batches |
| COUNT_TOKENS first | Estimate cost before processing large datasets |


## Key Takeaways

- Use `ACCOUNT_USAGE` views to track Cortex spending: functions, search, doc processing.
- `AI_COUNT_TOKENS` enables pre-processing cost estimates.
- Specialized functions (`AI_SENTIMENT`, `AI_CLASSIFY`) are cheaper than general `AI_COMPLETE`.
- Cache AI results in tables — don't recompute what hasn't changed.
- Cost monitoring is a key topic in Exam Domain 3.0.
